# Bronze -- VRA (Voo Regular Ativo)
Lê os 12 CSVs mensais do volume anac-services-analysis.bronze.files/vra/ e materializar anac-services-analysis.bronze.vra 

Regras da camada Bronze:
- *Nada de tipagem* -- tudo string, exatamente como veio o arquivo;
- *Nada de filtro* -- nenhuma linha descartada;
- *Colunas de audioria* -- de qual arquivo veio e quando foi integrado; 
- *Idempotente* -- rodar duas vezes não duplica. 

In [0]:
from pyspark.sql import functions as F

PATH = "/Volumes/anac_services_analysis/bronze/files/vra/*.csv"
TABLE = "anac_services_analysis.bronze.vra"

# Leitura
Quatro opções carregam quatro problemas de arquivo:
- sep=";" -- separador brasileito, não vírgula
- skipRow=1 -- a primeira linha é Atualizadp em: <data>, não o cabeçalho e o BOM EF BB BF mora nela, some junto
- header=true -- a segunda linha (a primeira que sobra) é o cabeçalho de verdade
- inferSchema desligado (default) -- bronze não tipa: tudo chega como string


In [0]:
row_data = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", True)
    .option("skipRows", 1)                       # Descarta "Atualizado em: ..." (E o BOM junto)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")                # Bronze não descarta linhas
    .load(PATH)
)

print("Colunas lidas do arquivo")
for c in row_data.columns:
    print(f" - {c}")
#

# Nomes de colunas: o Delta não aceita espaços

ICAO Empresa Aérea éum nome de coluna válido em CSV e inválido em Delta - espaço está na lista de caracteres proibidos ( ,;{}()\n\t=).
Então normalizamos o nome. Repare que isso não fere a regra da camada Bronze: o que o bronze preserva é o valor e a granularidade, não a grafia do cabeçalho. Nenhuma coluna é somada, removida, filtrada ou convertida.

O mapa fica explícito no código - nada de regexp_replace mágico, para que a correspondência com arquivo original seja auditável. 

In [0]:
import re
import unicodedata

def clean_csv_column_name(column_name: str) -> str:
    
    # 1. Normaliza texto para remover acentos
    # 2. Converte para minúsculas e remove hífens/caracteres no início
    # 3. Substitui caracteres especiais, parênteses e hífens por espaço
    # 4. Substitui espaços (simples ou múltiplos) por underline (_)
    # 5. Remove underlines sobressalentes no início ou fim
    
    col_name = (
        unicodedata.normalize("NFKD", column_name)
        .encode("ascii", "ignore")
        .decode("utf-8")
    )
    col_name = col_name.lower().strip()
    col_name = re.sub(r"[^\w\s]", " ", col_name)
    col_name = re.sub(r"\s+", "_", col_name)
    return col_name.strip("_")

dictNames = dict()
for i in row_data.columns:
    dictNames[i] = clean_csv_column_name(i)
    # print(f" - {i} -> {dictNames[i]}")

row_data_renamed = row_data.select(
    *[F.col(source).cast("string").alias(target) for source, target in dictNames.items()]
)

print(row_data_renamed)


# Auditoria
Duas colunas que o arquivo não tem e a tabela precisa ter:
- _source_file (de qual CSV a linha veio - _metadata é uma coluna oculta que o Spark expõe em qualque leitura de arquivo)
- _ingested_in.

In [0]:
bronze = row_data_renamed.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name")).withColumn("_ingerido_em", F.current_timestamp()
)

# Escrita idemmpotente

Estratégica: full refresh deterministico - mode("overwrite") sobre o conjunto inteiro de arquivos.

Por que essa e não um append com deduplicação:
- A fonte é imutável e completa: o volume tem os 12 arquivos de mês fechado, e a ANAC republica o mes inteiro quando corrige algo. A entradadefine o estado final - Logo o destino pode ser derivado inteiro dela. 
- Append exigiria uma chave de negócios para duplicar. O VRA não tem chave natrual única (o mesmo voo pode repetir legitimamente na mesma data - veja o código DI "Etapa de voo duplicado"). Duplicar no bronze seria decidir regras de negócio na camada errada.
- Overwriteno delta é atômico: ou a versão nova aparece inteira ou a antiga continua valeno. Ninguem lê tabela pela metade. 
- O histórico não se perde: cada overwrite gera uma nova versão. no log do delta, e a anteriror continua acessível por time travel (marco-04)

In [0]:
(
    bronze.write.format("delta")
    .mode("overwrite")    
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE)
)

print(f"{TABLE}: {spark.table(TABLE).count():,} linhas")

### Contextualização de Catálogo

In [0]:
spark.sql("""
    COMMENT ON TABLE {} IS 
        '
            Bronze - VRA - (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026). 
            Dado bruto: todas as colunas string, nenhuma linha descartada.
            Carga full refresh idempotente a partir de /Volums/anac_services_analysis/bronze/files/vra/.
        '           
""".format(TABLE))

### Demonstração dos dados

In [0]:
display(
    spark.sql(
        f"""
            SELECT _arquivo_origem, COUNT(*) as linhas, MAX(_ingerido_em) as data_ingerido
            FROM {TABLE}
            GROUP BY _arquivo_origem
            ORDER BY _arquivo_origem
        """
    )
)